# BirdCLEF 2026 - Inference Only Pipeline
This notebook is extracted from the main training pipeline. It performs ONLY inference and generates the submission.csv file using a pre-trained model.

**Instructions:**
- Add your trained model dataset to this notebook.
- Update `CFG.MODEL_PATH` to point to your `best_model.pth`.


In [ ]:
import os
import gc
import sys
import math
import time
import glob
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import librosa
import soundfile as sf
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import timm

import warnings
warnings.filterwarnings('ignore')

In [ ]:
class Config:
    ROOT_DIR = '/kaggle/input/competitions/birdclef-2026'
        
    TRAIN_CSV = os.path.join(ROOT_DIR, 'train.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')

    # Important: Point this to your trained model from the training output
    MODEL_PATH = '/kaggle/input/models/punyakdei/training-clef-multilabel/pytorch/default/2/best_model.pth'
    
    # Audio Setup
    SR = 32000
    WINDOW_SECONDS = 5
    
    # Mel Spectrogram Setup
    N_MELS = 128
    N_FFT = 2048
    HOP_LENGTH = 512
    FMIN = 20
    FMAX = 16000
    
    # Model Setup
    MODEL_NAME = 'tf_efficientnet_b0' 
    NUM_CLASSES = 0 # Will be populated automatically
    
CFG = Config()

# Load train.csv only for reporting; the model head must match the full
# competition schema to remain compatible with the new checkpoint.
print("Loading labels from train.csv and sample_submission.csv...")
df = pd.read_csv(CFG.TRAIN_CSV)
train_labels = sorted(df['primary_label'].unique())
sample_sub = pd.read_csv(os.path.join(CFG.ROOT_DIR, 'sample_submission.csv'))
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
unique_labels = submission_labels
CFG.NUM_CLASSES = len(unique_labels)
print(f"Detected {len(train_labels)} clip labels.")
print(f"Official submission labels: {CFG.NUM_CLASSES}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
class BirdModel(nn.Module):
    def __init__(self, model_name, num_classes, model_path=None, pretrained=False):
        super().__init__()

        if model_path is not None:
            self.backbone = timm.create_model(model_name, checkpoint_path=model_path, pretrained=pretrained, in_chans=3)
        else:
            self.backbone = timm.create_model(model_name, pretrained=pretrained, in_chans=3)
        
        if 'efficientnet' in model_name:
            in_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
        elif 'convnext' in model_name:
            in_features = self.backbone.head.fc.in_features
            self.backbone.head.fc = nn.Identity()
        else:
            in_features = self.backbone.get_classifier().in_features
            self.backbone.reset_classifier(0)
            
        self.head = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        out = self.head(features)
        return out

## Inference & Submission Engine

In [ ]:
# 1. Load Best Model
print("Loading best model for inference...")
model = BirdModel(CFG.MODEL_NAME, CFG.NUM_CLASSES).to(device)
try:
    model.load_state_dict(torch.load(CFG.MODEL_PATH, map_location=device))
    model.eval()
    print("Model loaded successfully.")
except Exception as e:
    print(f"Warning: Could not load '{CFG.MODEL_PATH}'. Check your file path. Error: {e}")

In [ ]:
# 2. Fallback Logic
TEST_DIR = os.path.join(CFG.ROOT_DIR, 'test_soundscapes')
test_files = []
if os.path.exists(TEST_DIR):
    test_files = sorted(glob.glob(f'{TEST_DIR}/*.ogg'))

if len(test_files) == 0:
    print('FALLBACK ACTIVE: No test files found. Using training soundscapes as dry-run.')
    test_files = sorted(glob.glob(f'{CFG.SOUNDSCAPE_DIR}/*.ogg'))[:5] # Use first 5 for speed
    IS_DRY_RUN = True
else:
    print(f'Found {len(test_files)} files in test directory.')
    IS_DRY_RUN = False

In [ ]:
# 3. Sliding Window Inference
mel_transform = T.MelSpectrogram(
    sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
    n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
).to(device)
amplitude_to_db = T.AmplitudeToDB(top_db=80).to(device)

all_predictions = []
all_row_ids = []

print(f"\n{'='*20} Starting Inference {'='*20}")
for audio_path in tqdm(test_files, desc="Processing Soundscapes"):
    filename = os.path.basename(audio_path).replace('.ogg', '')
    
    try:
        # Load full audio
        y, _ = sf.read(audio_path, always_2d=True)
        y = y.mean(axis=1) # Mono
    except Exception as e:
        print(f"Error reading {audio_path}: {e}")
        continue
        
    y_tensor = torch.tensor(y, dtype=torch.float32).to(device)
    total_samples = len(y_tensor)
    window_samples = CFG.SR * CFG.WINDOW_SECONDS
    
    # Calculate number of segments
    n_segments = math.ceil(total_samples / window_samples)
    
    for seg_idx in range(n_segments):
        start_sample = seg_idx * window_samples
        end_sample = start_sample + window_samples
        end_time_sec = (seg_idx + 1) * CFG.WINDOW_SECONDS
        row_id = f"{filename}_{end_time_sec}"
        
        # Extract and pad segment if needed
        segment = y_tensor[start_sample:end_sample]
        if len(segment) < window_samples:
            segment = F.pad(segment, (0, window_samples - len(segment)))
            
        # Transform
        with torch.no_grad():
            mel_spec = mel_transform(segment)
            mel_spec = amplitude_to_db(mel_spec)
            mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
            image = torch.stack([mel_spec, mel_spec, mel_spec]).unsqueeze(0) # Add batch dimension
            
            # Predict
            output = model(image)
            probs = torch.sigmoid(output).squeeze(0).cpu().numpy()
            
        all_row_ids.append(row_id)
        all_predictions.append(probs)

In [ ]:
# 4. Submission Formatting
print("\nFormatting submission...")

# The checkpoint now predicts the full competition label space directly.
prediction_df = pd.DataFrame(all_predictions, columns=submission_labels)
submission_df = prediction_df[['row_id'] + submission_labels] if 'row_id' in prediction_df.columns else prediction_df
submission_df.insert(0, 'row_id', all_row_ids)
submission_df = submission_df[['row_id'] + submission_labels]

# Basic sanity checks for Kaggle submission format.
expected_cols = len(submission_labels) + 1
if submission_df.shape[1] != expected_cols:
    raise ValueError(f"Submission has {submission_df.shape[1]} columns, expected {expected_cols}.")
if len(submission_df) != len(all_row_ids):
    raise ValueError(f"Submission has {len(submission_df)} rows, expected {len(all_row_ids)}.")
if submission_df.isnull().values.any():
    raise ValueError("Submission contains missing values.")

# Save to CSV
submission_path = 'submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved to {submission_path}")
print(f"Shape: {submission_df.shape}")
print(f"First few rows:")
display(submission_df.head(3))